# Online casino churn: 60-day survival forecast

This notebook runs the whole project end to end, from synthetic data to the scoring of
the current player base. The reusable logic lives in the `churn` package
(`generate_data`, `features`, `models`, `evaluation`, `pipeline`); the notebook calls it
step by step and shows every intermediate result.

Sections are numbered after the project requirements:

- **Data** and **Problem setup**: the generation script, churn labels, features, split
- **2-3** Cox and AFT models, tuned on the validation integrated Brier score
- **4** test: predicted vs observed, **5** hazard: predicted vs observed
- **6** model quality and the best model, **7** Brier score and AUC over time
- **8** calibration, **9** bootstrap of the metrics
- **10** feature importance, **11** dependence plots for all feature pairs
- **12** remaining useful life vs Kaplan-Meier, **13** backtest
- **14** production model on train + validation, **15** evaluation vs production
- **16** riskiest players, **17** their scoring
- **18** artifacts for the dashboard, **19** the dashboard

A full run takes about two minutes. It rewrites `data/` and `artifacts/` with the same
files that `python -m churn.generate_data` and `python -m churn.pipeline` produce.

In [1]:
import warnings

# Optuna imports tqdm.auto, which warns when ipywidgets is missing; no progress bar is used.
warnings.filterwarnings("ignore", message="IProgress not found")

In [2]:
import json
from datetime import UTC, datetime

import numpy as np
import optuna
import pandas as pd
import plotly.graph_objects as go
from IPython.display import display
from numpy.typing import ArrayLike
from plotly.subplots import make_subplots
from scipy.stats import spearmanr

from churn import config, features, generate_data
from churn import evaluation as ev
from churn.models import MODELS
from churn.pipeline import (
    backtest,
    describe,
    diagnostics,
    explain,
    make_model,
    score_players,
    tune,
    xy,
)

N_TRIALS = 40  # Optuna trials per model
N_BOOTSTRAP = 200  # bootstrap resamples of the test metrics
SEED = config.SEED
DATA_DIR = config.DATA_DIR  # rewritten by the data step
ARTIFACTS_DIR = config.ARTIFACTS_DIR  # rewritten by step 18

TIMES = config.TIME_GRID  # days 1..85 on which survival curves are evaluated
HORIZON = config.HORIZON_DAYS
K_HORIZON = int(np.flatnonzero(TIMES == HORIZON)[0])  # column of the horizon in a curve matrix
FEATURES = list(config.FEATURES)

optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option("display.precision", 4, "display.max_columns", 40)

In [3]:
GREY = "#6b7280"
COLORS = {
    "cox": "#2563eb",
    "aft": "#f97316",
    "production": "#16a34a",
    "all": GREY,
    "low": "#16a34a",
    "medium": "#f59e0b",
    "high": "#dc2626",
}
DASHES = {"raw": "dot", "calibrated": "solid"}


def line(
    x: ArrayLike,
    y: ArrayLike,
    name: str,
    color: str,
    dash: str = "solid",
    width: float = 2,
    **kwargs,
) -> go.Scatter:
    """Line trace."""
    style = {"color": color, "dash": dash, "width": width}
    return go.Scatter(x=x, y=y, name=name, mode="lines", line=style, **kwargs)


def show(
    fig: go.Figure,
    title: str,
    x_title: str | None = None,
    y_title: str | None = None,
    height: int = 420,
    **layout,
) -> None:
    """Apply the common layout and display the figure; ``None`` keeps an axis title."""
    titles = {"xaxis_title": x_title, "yaxis_title": y_title}
    fig.update_layout(
        title=title,
        template="plotly_white",
        height=height,
        margin={"l": 10, "r": 10, "t": 70, "b": 10},
        **{key: value for key, value in titles.items() if value is not None},
        **layout,
    )
    fig.show()

## Data: generation script

`churn/generate_data.py` simulates the players of an online casino day by day. Every
player moves through hidden states (engaged, fading, gone, with occasional win-backs).
The daily chance to start fading depends on the acquisition channel, the favourite game,
age, a latent engagement level, tenure, recent luck and recent bonuses. Only behaviour is
exported, so the models have to infer the hidden state from it.

The same script runs from the command line: `uv run python -m churn.generate_data`.

In [4]:
players, activity = generate_data.generate(seed=SEED)
DATA_DIR.mkdir(parents=True, exist_ok=True)
players.to_parquet(DATA_DIR / "players.parquet", index=False)
activity.to_parquet(DATA_DIR / "activity.parquet", index=False)

# Read the files back: the rest of the notebook works on what is stored on disk.
players, activity = features.load_raw(DATA_DIR)
first_day, last_day = activity["date"].min().date(), activity["date"].max().date()
print(f"{len(players):,} players, {len(activity):,} player-day rows, {first_day} .. {last_day}")
display(players.head(), activity.head())

20,000 players, 1,700,671 player-day rows, 2024-01-01 .. 2026-06-30


,player_id,registration_date,age,acquisition_channel,favorite_game
0,1,2024-03-22,23,organic,crash_games
1,2,2025-12-06,50,affiliate,slots
2,3,2025-08-19,41,affiliate,slots
3,4,2025-02-04,38,affiliate,table_games
4,5,2025-01-29,42,affiliate,table_games


,player_id,date,sessions,bets,turnover,ggr,deposits,withdrawals,bonus
0,2316,2024-01-01,1,73,33.61,16.18,20.93,0.0,0.0
1,2442,2024-01-01,1,70,90.95,28.54,109.15,0.0,0.0
2,3674,2024-01-01,1,42,19.73,0.79,16.66,0.0,0.0
3,5083,2024-01-01,1,26,50.00,-4.35,71.26,0.0,0.0
4,5760,2024-01-01,2,126,221.08,79.49,99.59,0.0,0.0


In [5]:
daily = activity.groupby("date").size()
fig = go.Figure(line(daily.index, daily, "active players", COLORS["cox"]))
show(fig, "Players active per day", "date", "players", height=300)

## Problem setup: churn, snapshots, labels, temporal split

- **Churn** is the first run of 30 consecutive days without activity. It is dated on the
  30th inactive day, the first moment it can be known without looking into the future.
  A player leaves the population after the first churn.
- **Snapshots.** On the first day of every month each player still at risk becomes one
  row. Features use activity strictly before the snapshot date; the target is the number
  of days from the snapshot to churn.
- **Censoring.** Labels are read on a cut-off date and followed for at most 90 days. A
  churn on the very last day of follow-up counts as censored, so every event lies
  strictly inside the follow-up window, which the IPCW-based metrics need.
- **Temporal split.** Train: snapshots 2024-07-01 .. 2025-12-01 with labels as known on
  2026-01-01. Validation: snapshot 2026-01-01. Test: snapshot 2026-04-01. Scoring:
  snapshot 2026-07-01, no labels yet. Validation and test hold one row per player.
- **Target.** `P(churn within 60 days) = 1 - S(60 | x)`; survival curves are evaluated on
  the daily grid 1..85, which stays inside the 90-day follow-up.

In [6]:
snapshots = features.build_snapshots(players, activity, config.ALL_SNAPSHOTS)
data_cutoff = config.DATA_END + pd.Timedelta(days=1)  # first day without data
date = snapshots["snapshot_date"]
# Each split reads its labels on the date the next one starts, so nothing leaks backwards.
train = features.add_labels(snapshots[date.isin(config.TRAIN_SNAPSHOTS)], config.VAL_SNAPSHOT)
val = features.add_labels(snapshots[date == config.VAL_SNAPSHOT], config.TEST_SNAPSHOT)
test = features.add_labels(snapshots[date == config.TEST_SNAPSHOT], data_cutoff)
scoring = snapshots[date == config.SCORING_SNAPSHOT]
X_val, t_val, e_val = xy(val)
X_test, t_test, e_test = xy(test)

overview = pd.DataFrame.from_dict(
    {
        "train": describe(train),
        "val": describe(val, full_follow_up=True),
        "test": describe(test, full_follow_up=True),
        "scoring": {"rows": len(scoring)},
    },
    orient="index",
)
counts = ["rows", "players", "snapshots"]
overview[counts] = overview[counts].astype("Int64")
display(overview, test[["player_id", "snapshot_date", *FEATURES, "time", "event"]].head())

,rows,players,snapshots,event_rate,churn_rate_horizon
train,63229,13852,18,0.3481,NaN
val,4235,4235,1,0.3450,0.268
test,4201,4201,1,0.3325,0.258
scoring,4328,<NA>,<NA>,NaN,NaN


,player_id,snapshot_date,tenure_days,recency_days,active_days_7d,active_days_30d,activity_ratio_7d_30d,sessions_30d,turnover_30d,ggr_30d,deposits_30d,withdrawals_30d,bonus_30d,avg_stake_30d,age,acquisition_channel,favorite_game,time,event
75880,2,2026-04-01,116,1,6,27,0.9524,58,4437.81,327.48,295.52,84.41,11.64,1.2568,50,affiliate,slots,90.0,False
75881,6,2026-04-01,38,1,3,15,0.8571,30,2217.96,101.30,450.31,115.06,28.04,1.8422,23,organic,crash_games,90.0,False
75882,8,2026-04-01,186,1,5,23,0.9317,45,601.01,-52.00,77.76,14.05,7.40,0.2191,30,affiliate,slots,90.0,False
75883,16,2026-04-01,105,1,2,14,0.6122,24,590.60,-31.03,96.53,32.73,7.46,0.3977,26,organic,slots,90.0,False
75884,19,2026-04-01,56,1,6,23,1.1180,52,7093.09,-143.11,884.00,102.41,282.32,5.5501,25,affiliate,live_casino,90.0,False


## 2-3. XGBoost Cox and AFT, tuned on the validation integrated Brier score

- **Cox** (`survival:cox`): the booster outputs a log hazard ratio `eta(x)`. With the
  Breslow baseline cumulative hazard `H0` estimated on the training data,
  `S(t | x) = exp(-H0(t) * exp(eta(x)))`.
- **AFT** (`survival:aft`): `log T = mu(x) + sigma * Z`, so
  `S(t | x) = P(Z > (log t - mu(x)) / sigma)`. The noise distribution and `sigma` are
  tuned.

Both models expose `predict_survival(X, times)`, so everything downstream is shared.
Optuna (TPE) minimises `integrated_brier_score` on the validation snapshot. The number of
boosting rounds is tuned directly instead of using early stopping, so the production
refit in step 14, which has no validation set left, can reuse it.

In [7]:
tuned, models, val_surv, trials = {}, {}, {}, []
for name in MODELS:
    tuned[name], model_trials = tune(name, train, val, N_TRIALS, SEED)
    trials.append(model_trials)
    models[name] = make_model(name, tuned[name], SEED).fit(*xy(train))
    val_surv[name] = models[name].predict_survival(X_val, TIMES)

tables = {"tuning_trials": pd.concat(trials, ignore_index=True)}  # all saved in step 18
quality = [
    {
        "split": "val",
        "model": name,
        "variant": "raw",
        **ev.summary_metrics(t_val, e_val, surv, TIMES, HORIZON),
    }
    for name, surv in val_surv.items()
]
val_ibs = {row["model"]: row["ibs"] for row in quality}

best_params = pd.DataFrame(tuned).map(
    lambda v: "" if pd.isna(v) else f"{v:.4g}" if isinstance(v, float) else str(v)
)
display(best_params.rename_axis("hyperparameter"), pd.DataFrame(quality))

,cox,aft
hyperparameter,,
num_boost_round,375,400
learning_rate,0.07125,0.1847
max_depth,6,2
min_child_weight,11.29,17.35
subsample,0.9251,0.6393
colsample_bytree,0.8959,0.7714
lambda,21.78,0.2042
max_delta_step,0.4702,
aft_loss_distribution,,normal


,split,model,variant,c_index,ibs,brier_horizon,auc_horizon
0,val,cox,raw,0.8335,0.0685,0.1058,0.8745
1,val,aft,raw,0.8357,0.0696,0.1066,0.8735


In [8]:
fig = go.Figure()
for name, part in tables["tuning_trials"].groupby("model", sort=False):
    marker = {"color": COLORS[name], "opacity": 0.4}
    points = go.Scatter(
        x=part["trial"], y=part["ibs"], mode="markers", name=f"{name}: trial", marker=marker
    )
    fig.add_trace(points)
    fig.add_trace(line(part["trial"], part["best_ibs"], f"{name}: best so far", COLORS[name]))
show(fig, "Optuna search on validation", "trial", "IBS (log scale)", yaxis_type="log")

Requirement 8 asks for calibrated models. The calibrators are fitted here, on the
validation snapshot, because steps 4-7 report raw and calibrated predictions side by
side; step 8 looks at the calibration itself.

At every day `t` of the grid an isotonic regression maps the predicted `1 - S(t | x)`
onto the observed frequency of churn by day `t`; the calibrated curves are then made
non-increasing in `t`. No IPCW is needed: on validation nobody is censored before the
end of the grid, which `fit` checks.

In [9]:
calibrators, test_surv = {}, {}
for name, model in models.items():
    calibrators[name] = ev.IsotonicSurvivalCalibrator().fit(val_surv[name], t_val, e_val, TIMES)
    raw = model.predict_survival(X_test, TIMES)
    test_surv[(name, "raw")] = raw
    test_surv[(name, "calibrated")] = calibrators[name].transform(raw, TIMES)
variant_titles = [f"{name} {variant}" for name, variant in test_surv]

## 4. Test: predicted vs observed

Three views of the same question, on the untouched test snapshot:

- the mean predicted survival curve against the Kaplan-Meier estimate of what happened;
- the mean predicted churn probability against the observed churn rate at 7, 14, 30 and
  60 days;
- predicted against observed 60-day churn by decile of predicted risk.

In [10]:
tables |= diagnostics(t_test, e_test, test_surv)

curves = tables["survival_curves"]
observed = curves.drop_duplicates("day")
fig = go.Figure(
    line(observed["day"], observed["observed"], "observed (Kaplan-Meier)", GREY, width=5)
)
for (name, variant), part in curves.groupby(["model", "variant"], sort=False):
    label = f"{name} {variant}"
    fig.add_trace(line(part["day"], part["predicted"], label, COLORS[name], DASHES[variant]))
show(
    fig,
    "Test: mean predicted survival vs Kaplan-Meier",
    "days since snapshot",
    "share of players not churned",
    yaxis_tickformat=".0%",
)

In [11]:
at_horizons = np.isin(TIMES, config.SCORE_HORIZONS)
rates = {
    f"{name} {variant}": 1 - surv[:, at_horizons].mean(axis=0)
    for (name, variant), surv in test_surv.items()
}
rates["observed"] = [(e_test & (t_test <= h)).mean() for h in config.SCORE_HORIZONS]
display(pd.DataFrame(rates, index=[f"churn by day {h}" for h in config.SCORE_HORIZONS]).T)

,churn by day 7,churn by day 14,churn by day 30,churn by day 60
cox raw,0.0357,0.0773,0.1656,0.2741
cox calibrated,0.0350,0.0696,0.1490,0.2633
aft raw,0.0399,0.0715,0.1326,0.2446
aft calibrated,0.0354,0.0679,0.1475,0.2614
observed,0.0364,0.0714,0.1485,0.2580


In [12]:
deciles = tables["calibration"].query("horizon == @HORIZON")
fig = make_subplots(rows=1, cols=4, subplot_titles=variant_titles, shared_yaxes=True)
for col, (name, variant) in enumerate(test_surv, start=1):
    part = deciles[(deciles["model"] == name) & (deciles["variant"] == variant)]
    fig.add_trace(go.Bar(x=part["bin"], y=part["predicted"], marker_color=COLORS[name]), 1, col)
    fig.add_trace(go.Bar(x=part["bin"], y=part["observed"], marker_color=GREY), 1, col)
fig.update_xaxes(title_text="risk decile")
fig.update_yaxes(tickformat=".0%")
show(
    fig,
    f"Test: churn within {HORIZON} days by risk decile, predicted (colour) vs observed (grey)",
    height=360,
    showlegend=False,
)

## 5. Hazard: predicted vs observed

The observed hazard on day `t` is the number of players who churned on that day divided
by the number still at risk. The predicted hazard is the one of the group's mean
predicted curve, `1 - S(t) / S(t - 1)`, which is what the observed ratio estimates in a
heterogeneous group. Groups are terciles of the predicted 60-day churn probability.

A player last active `r` days before the snapshot cannot churn before day `31 - r`, so a
group of recently active players has no hazard at all during the first weeks.

In [13]:
hazard = tables["hazard"]
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=variant_titles,
    shared_xaxes=True,
    shared_yaxes=True,
    vertical_spacing=0.1,
)
for position, (name, variant) in enumerate(test_surv):
    row, col = position // 2 + 1, position % 2 + 1
    part = hazard[(hazard["model"] == name) & (hazard["variant"] == variant)]
    for group, days in part.groupby("group", sort=False):
        smooth = days[["predicted", "observed"]].rolling(7, center=True, min_periods=1).mean()
        color, legend = COLORS[group], {"legendgroup": group, "showlegend": position == 0}
        fig.add_trace(line(days["day"], smooth["predicted"], group, color, **legend), row, col)
        legend["showlegend"] = False
        fig.add_trace(
            line(days["day"], smooth["observed"], group, color, "dot", **legend), row, col
        )
fig.update_xaxes(title_text="days since snapshot", row=2)
fig.update_yaxes(tickformat=".1%")
show(
    fig,
    "Test: daily churn hazard by risk group, predicted (solid) vs observed (dotted), "
    "7-day moving average",
    height=620,
)

## 6. Model quality and the best model

- `c_index`: Harrell's concordance of the 60-day churn probability with the durations.
- `ibs`: integrated Brier score over the grid (lower is better), the tuning target.
- `brier_horizon`, `auc_horizon`: IPCW Brier score and cumulative/dynamic AUC at 60 days.

The `kaplan_meier` row is a reference without covariates: the Kaplan-Meier curve of the
training data assigned to every test player. The best model is chosen by validation IBS
only, so the test numbers stay an unbiased report.

In [14]:
km_train = ev.km_on_grid(train["time"].to_numpy(), train["event"].to_numpy(), TIMES)
reference = {("kaplan_meier", "raw"): np.tile(km_train, (len(test), 1))}
for (name, variant), surv in (test_surv | reference).items():
    metrics = ev.summary_metrics(t_test, e_test, surv, TIMES, HORIZON)
    quality.append({"split": "test", "model": name, "variant": variant, **metrics})
tables["model_quality"] = pd.DataFrame(quality)

best = min(val_ibs, key=val_ibs.get)
display(tables["model_quality"])
print(f"Validation IBS: {val_ibs}")
print(f"Best model by validation IBS: {best}")

,split,model,variant,c_index,ibs,brier_horizon,auc_horizon
0,val,cox,raw,0.8335,0.0685,0.1058,0.8745
1,val,aft,raw,0.8357,0.0696,0.1066,0.8735
2,test,cox,raw,0.8199,0.0719,0.1093,0.8624
3,test,cox,calibrated,0.8204,0.0714,0.1094,0.8617
4,test,aft,raw,0.8234,0.0724,0.1091,0.8610
5,test,aft,calibrated,0.8209,0.0714,0.1102,0.8589
6,test,kaplan_meier,raw,0.5000,0.1446,0.1922,0.5000


Validation IBS: {'cox': 0.06846798557968878, 'aft': 0.06955828108358686}
Best model by validation IBS: cox


## 7. Brier score and AUC over time

Both metrics are computed at every day of the grid with inverse-probability-of-censoring
weights. The dashed line marks the 60-day business horizon.

In [15]:
over_time = tables["metrics_over_time"]
titles = ["Brier score (lower is better)", "cumulative/dynamic AUC (higher is better)"]
fig = make_subplots(rows=1, cols=2, subplot_titles=titles)
for (name, variant), part in over_time.groupby(["model", "variant"], sort=False):
    label = f"{name} {variant}"
    for col, metric in enumerate(("brier", "auc"), start=1):
        legend = {"legendgroup": label, "showlegend": col == 1}
        trace = line(part["day"], part[metric], label, COLORS[name], DASHES[variant], **legend)
        fig.add_trace(trace, 1, col)
fig.add_vline(x=HORIZON, line_dash="dash", line_color=GREY)
fig.update_xaxes(title_text="days since snapshot")
show(fig, "Test: time-dependent Brier score and AUC")

selected = over_time[over_time["day"].isin(config.SCORE_HORIZONS)]
display(selected.pivot(index=["model", "variant"], columns="day", values=["brier", "auc"]))

brier                             auc                  \
day                   7       14      30      60      7       14      30   
model variant                                                              
aft   calibrated  0.0045  0.0074  0.0446  0.1102  0.9895  0.9986  0.9573   
      raw         0.0077  0.0124  0.0459  0.1091  0.9991  0.9986  0.9593   
cox   calibrated  0.0043  0.0072  0.0454  0.1094  0.9991  0.9952  0.9548   
      raw         0.0042  0.0084  0.0456  0.1093  0.9992  0.9985  0.9575   

                          
day                   60  
model variant             
aft   calibrated  0.8589  
      raw         0.8610  
cox   calibrated  0.8617  
      raw         0.8624

## 8. Calibration

The calibrators were fitted on validation right after tuning. Here is what they do:

- the isotonic map at the 60-day horizon (raw probability in, calibrated probability out);
- reliability on test before and after: predicted against observed churn rate by risk
  decile, where a perfectly calibrated model lies on the diagonal;
- the mean absolute gap between predicted and observed churn rate across the deciles, at
  every scoring horizon.

Whether calibration changes the metrics by more than noise is tested in step 9.

In [16]:
fig = go.Figure(line([0, 1], [0, 1], "identity", GREY, "dash"))
for name, calibrator in calibrators.items():
    raw_probability, calibrated_probability = calibrator.knots_[K_HORIZON]
    fig.add_trace(line(raw_probability, calibrated_probability, name, COLORS[name]))
show(
    fig,
    f"Isotonic calibration maps at {HORIZON} days (fitted on validation)",
    "raw P(churn)",
    "calibrated P(churn)",
    height=380,
)

In [17]:
fig = make_subplots(rows=1, cols=len(MODELS), subplot_titles=list(MODELS), shared_yaxes=True)
for col, name in enumerate(MODELS, start=1):
    fig.add_trace(line([0, 1], [0, 1], "perfect", GREY, "dash", showlegend=col == 1), 1, col)
    for variant, dash in DASHES.items():
        part = deciles[(deciles["model"] == name) & (deciles["variant"] == variant)]
        curve = go.Scatter(
            x=part["predicted"],
            y=part["observed"],
            mode="lines+markers",
            name=f"{name} {variant}",
            line={"color": COLORS[name], "dash": dash},
        )
        fig.add_trace(curve, 1, col)
fig.update_xaxes(title_text="predicted churn rate", tickformat=".0%")
fig.update_yaxes(tickformat=".0%")
show(fig, f"Test: reliability at {HORIZON} days by risk decile", y_title="observed churn rate")

gap = tables["calibration"].assign(gap=lambda d: (d["predicted"] - d["observed"]).abs())
gap = gap.pivot_table(index=["model", "variant"], columns="horizon", values="gap", aggfunc="mean")
display(gap.rename_axis(columns="mean |predicted - observed| at day"))

mean |predicted - observed| at day      7       14      30      60
model variant                                                     
aft   calibrated                    0.0010  0.0035  0.0111  0.0159
      raw                           0.0035  0.0185  0.0185  0.0165
cox   calibrated                    0.0014  0.0018  0.0127  0.0223
      raw                           0.0014  0.0108  0.0232  0.0205

## 9. Bootstrap of the metrics

Test players are resampled with replacement (one row per player, so rows are
independent). The same resample is used for every model, which makes the differences
paired: an interval that excludes zero means the difference is larger than what
resampling noise explains.

In [18]:
named = {f"{name}_{variant}": surv for (name, variant), surv in test_surv.items()}
draws = ev.bootstrap_metrics(t_test, e_test, named, TIMES, HORIZON, N_BOOTSTRAP, SEED)

point = tables["model_quality"].query("split == 'test' and model != 'kaplan_meier'")
point = point.assign(model=point["model"] + "_" + point["variant"]).melt(
    id_vars="model", value_vars=list(ev.METRICS), var_name="metric", value_name="point"
)
tables["bootstrap_summary"] = ev.bootstrap_summary(draws).merge(point, on=["model", "metric"])
pairs = [("cox_raw", "aft_raw"), ("cox_calibrated", "aft_calibrated")]
pairs += [(f"{name}_calibrated", f"{name}_raw") for name in MODELS]
tables["bootstrap_differences"] = pd.concat(
    [ev.bootstrap_difference(draws, a, b) for a, b in pairs], ignore_index=True
)
tables["bootstrap_draws"] = draws

In [19]:
summary = tables["bootstrap_summary"]
fig = make_subplots(
    rows=1, cols=len(ev.METRICS), subplot_titles=list(ev.METRICS), shared_yaxes=True
)
for col, metric in enumerate(ev.METRICS, start=1):
    part = summary[summary["metric"] == metric]
    interval = {
        "type": "data",
        "symmetric": False,
        "array": part["ci_high"] - part["point"],
        "arrayminus": part["point"] - part["ci_low"],
        "color": GREY,
    }
    marker = {"size": 10, "color": [COLORS[key.rsplit("_", 1)[0]] for key in part["model"]]}
    estimate = go.Scatter(
        x=part["point"], y=part["model"], mode="markers", marker=marker, error_x=interval
    )
    fig.add_trace(estimate, 1, col)
show(
    fig,
    f"Test metrics with 95% bootstrap intervals ({N_BOOTSTRAP} resamples of players)",
    height=320,
    showlegend=False,
)

differences = tables["bootstrap_differences"]
excludes_zero = (differences["ci_low"] > 0) | (differences["ci_high"] < 0)
readable = differences.assign(excludes_zero=excludes_zero)
bounds = ["mean", "ci_low", "ci_high"]
readable[bounds] = readable[bounds].map("{:+.5f}".format)  # the differences are tiny
display(summary, readable)

,model,metric,mean,ci_low,ci_high,point
0,cox_raw,c_index,0.8197,0.8072,0.8299,0.8199
1,cox_raw,ibs,0.0719,0.0677,0.0756,0.0719
2,cox_raw,brier_horizon,0.1093,0.1027,0.1159,0.1093
3,cox_raw,auc_horizon,0.8621,0.8469,0.8737,0.8624
4,cox_calibrated,c_index,0.8203,0.8077,0.8307,0.8204
5,cox_calibrated,ibs,0.0714,0.0671,0.0752,0.0714
6,cox_calibrated,brier_horizon,0.1094,0.1025,0.1160,0.1094
7,cox_calibrated,auc_horizon,0.8614,0.8461,0.8730,0.8617
8,aft_raw,c_index,0.8233,0.8102,0.8334,0.8234
9,aft_raw,ibs,0.0723,0.0684,0.0759,0.0724


,model,metric,mean,ci_low,ci_high,excludes_zero
0,cox_raw - aft_raw,auc_horizon,+0.00144,-0.00089,+0.00398,False
1,cox_raw - aft_raw,brier_horizon,+0.00024,-0.00079,+0.00133,False
2,cox_raw - aft_raw,c_index,-0.00356,-0.00538,-0.00155,True
3,cox_raw - aft_raw,ibs,-0.00045,-0.00112,+0.00032,False
4,cox_calibrated - aft_calibrated,auc_horizon,+0.00274,+0.00013,+0.00516,True
5,cox_calibrated - aft_calibrated,brier_horizon,-0.00079,-0.00191,+0.00044,False
6,cox_calibrated - aft_calibrated,c_index,-0.00057,-0.00255,+0.00169,False
7,cox_calibrated - aft_calibrated,ibs,+0.00004,-0.00062,+0.00062,False
8,cox_calibrated - cox_raw,auc_horizon,-0.00076,-0.00178,+0.00027,False
9,cox_calibrated - cox_raw,brier_horizon,+0.00005,-0.00043,+0.00065,False


## 10. Feature importance

Two measures for every model:

- `gain_share`: the share of the total split gain that goes to a feature;
- `mean_abs_shap`: the mean absolute TreeSHAP contribution on a sample of test players.
  Contributions are oriented so that a positive value pushes towards churn.

Cox explains a log hazard and AFT a log time, so the SHAP magnitudes are shown as shares
within a model. The production model joins the comparison in step 14.

In [20]:
explained = {name: explain(model, name, X_test, SEED) for name, model in models.items()}


def importance_chart(importance: pd.DataFrame, title: str) -> None:
    """Grouped bars of the two importance measures, one bar group per model."""
    kinds = {"mean_abs_shap": "share of mean |SHAP|", "gain_share": "share of split gain"}
    shares = importance.copy()
    shares["mean_abs_shap"] /= shares.groupby("model")["mean_abs_shap"].transform("sum")
    order = shares.groupby("feature")["mean_abs_shap"].mean().sort_values().index
    fig = make_subplots(rows=1, cols=2, subplot_titles=list(kinds.values()), shared_yaxes=True)
    for name, part in shares.groupby("model", sort=False):
        part = part.set_index("feature").loc[order]
        for col, kind in enumerate(kinds, start=1):
            bars = go.Bar(
                y=order,
                x=part[kind],
                name=name,
                orientation="h",
                marker_color=COLORS[name],
                legendgroup=name,
                showlegend=col == 1,
            )
            fig.add_trace(bars, 1, col)
    fig.update_xaxes(tickformat=".0%")
    show(fig, title, height=520, barmode="group")


importance = pd.concat([table for table, _ in explained.values()], ignore_index=True)
importance_chart(importance, "Feature importance of the evaluation models (test sample)")
wide = importance.pivot(index="feature", columns="model", values=["gain_share", "mean_abs_shap"])
display(wide.sort_values(("mean_abs_shap", best), ascending=False))

gain_share         mean_abs_shap        
model                        aft     cox           aft     cox
feature                                                       
recency_days              0.6905  0.5400        0.4350  1.2013
active_days_30d           0.0508  0.0451        0.1246  0.3046
active_days_7d            0.0527  0.1737        0.2084  0.2840
acquisition_channel       0.0154  0.0193        0.1225  0.2439
tenure_days               0.0132  0.0368        0.0665  0.1215
activity_ratio_7d_30d     0.1342  0.0535        0.1123  0.1148
age                       0.0056  0.0181        0.0434  0.0845
sessions_30d              0.0117  0.0235        0.0328  0.0825
bonus_30d                 0.0050  0.0075        0.0434  0.0763
favorite_game             0.0043  0.0075        0.0344  0.0669
ggr_30d                   0.0050  0.0204        0.0363  0.0647
avg_stake_30d             0.0026  0.0196        0.0367  0.0525
turnover_30d              0.0030  0.0157        0.0169  0.0471
deposits_30d              0.0029  0.0121        0.0316  0.0438
withdrawals_30d           0.0030  0.0072        0.0179  0.0343

## 11. Dependence plots for all feature pairs

A SHAP dependence plot shows the value of a feature on the x axis and its contribution to
the risk score on the y axis; colouring the points by a second feature reveals
interactions. Below there is one figure per feature with one panel per colouring
feature, which covers every ordered pair of features.

Colour runs from dark (low value of the panel's feature) to yellow (high value), clipped
to the 5th-95th percentile. Categorical features are drawn at integer positions with a
little horizontal jitter. Each panel shows a random subset of the explained sample to
keep the notebook light.

In [21]:
DEPENDENCE_MODELS = [best]  # add the other key of `models` to draw it as well
DEPENDENCE_POINTS = 500  # points per panel


def positions(values: pd.Series) -> tuple[np.ndarray, list[str] | None]:
    """Numeric plotting positions and, for categorical data, the category names."""
    if isinstance(values.dtype, pd.CategoricalDtype):
        return values.cat.codes.to_numpy(dtype="float32"), list(values.cat.categories)
    return values.to_numpy(dtype="float32"), None


def dependence_grid(shap: pd.DataFrame, feature: str) -> go.Figure:
    """SHAP dependence of ``feature``, one panel per colouring feature."""
    sample = shap.sample(min(len(shap), DEPENDENCE_POINTS), random_state=SEED)
    x, categories = positions(sample[feature])
    if categories:
        x = x + np.random.default_rng(SEED).uniform(-0.25, 0.25, len(x)).astype("float32")
    y = sample[f"shap_{feature}"].to_numpy(dtype="float32")
    others = [name for name in FEATURES if name != feature]
    n_cols = 5
    n_rows = -(-len(others) // n_cols)
    fig = make_subplots(
        rows=n_rows,
        cols=n_cols,
        subplot_titles=others,
        shared_xaxes=True,
        shared_yaxes=True,
        vertical_spacing=0.08,
        horizontal_spacing=0.02,
    )
    for position, other in enumerate(others):
        color, names = positions(sample[other])
        low, high = (0, len(names) - 1) if names else np.nanpercentile(color, [5, 95])
        marker = {"size": 4, "opacity": 0.7, "color": color, "colorscale": "Viridis"}
        marker |= {"cmin": float(low), "cmax": float(high)}
        points = go.Scatter(x=x, y=y, mode="markers", marker=marker, hoverinfo="skip")
        fig.add_trace(points, position // n_cols + 1, position % n_cols + 1)
    # Shared x axes label only the bottom row; columns without a bottom panel need their own.
    last_row_panels = len(others) % n_cols or n_cols
    for col in range(last_row_panels + 1, n_cols + 1):
        fig.update_xaxes(showticklabels=True, row=n_rows - 1, col=col)
    if categories:
        fig.update_xaxes(tickvals=list(range(len(categories))), ticktext=categories)
    fig.update_annotations(font_size=11)  # panel titles are long feature names
    return fig


for name in DEPENDENCE_MODELS:
    for feature in FEATURES:
        title = f"{name}: SHAP value of {feature} against its value, coloured by the panel feature"
        show(dependence_grid(explained[name][1], feature), title, height=600, showlegend=False)

## 12. Remaining useful life: best model vs Kaplan-Meier

Remaining useful life (RUL) is the expected number of days until churn. The grid ends at
day 85, so it is the restricted mean `E[min(T, 85)] = 1 + S(1) + ... + S(84)`.

- **Model**: computed from each player's calibrated survival curve.
- **Kaplan-Meier baseline**: the same number for everybody, from the Kaplan-Meier curve
  of the training data. This is what one can say without a model.
- **Observed**: within each decile of predicted RUL, the restricted mean of the
  Kaplan-Meier curve of the test players in that decile.

The summary compares both predictions with the realised `min(T, 85)` of every player.

In [22]:
tables["rul"], rul_summary = ev.rul_tables(
    t_test, e_test, test_surv[(best, "calibrated")], TIMES, km_train
)
rul = tables["rul"]
fig = go.Figure(
    [
        go.Bar(
            x=rul["bin"], y=rul["predicted"], name=f"{best} calibrated", marker_color=COLORS[best]
        ),
        go.Bar(x=rul["bin"], y=rul["observed"], name="observed in the decile", marker_color=GREY),
    ]
)
baseline = line(rul["bin"], rul["km_baseline"], "Kaplan-Meier baseline", COLORS["high"], "dash")
fig.add_trace(baseline)
show(
    fig,
    "Test: remaining useful life by decile of predicted RUL",
    "decile of predicted RUL (1 = shortest)",
    f"expected days to churn, capped at {TIMES[-1]}",
)
display(pd.Series(rul_summary, name="value").to_frame(), rul)

,value
restricted_to_days,85.0000
mean_predicted,69.1031
mean_observed,69.3411
km_baseline,67.8575
mae_model,10.2721
mae_km,22.4506
rmse_model,15.9010
rmse_km,26.3444


,bin,n,predicted,observed,km_baseline
0,1,421,12.9605,14.2067,67.8575
1,2,420,48.2794,50.2381,67.8575
2,3,420,68.2242,70.1571,67.8575
3,4,420,74.0422,74.4643,67.8575
4,5,420,78.1368,77.6214,67.8575
5,6,420,80.4473,79.5762,67.8575
6,7,420,81.3131,80.3857,67.8575
7,8,420,81.5785,80.9500,67.8575
8,9,420,82.4471,82.1857,67.8575
9,10,420,83.7358,83.7571,67.8575


## 13. Backtest

Rolling origin over the monthly snapshots 2025-07-01 .. 2026-05-01. For every date both
models are refitted on all earlier snapshots, with labels as they were known on that
date, and then score the players at risk on that date. The forecast is compared with
what happened in the following 60 days.

Hyperparameters are the ones tuned once on the main split (a look-ahead for the early
folds) and predictions are not calibrated.

In [23]:
tables["backtest"] = backtest(snapshots, tuned, data_cutoff, SEED)
folds = tables["backtest"]

titles = [f"churn rate within {HORIZON} days", f"AUC at {HORIZON} days"]
fig = make_subplots(rows=1, cols=2, subplot_titles=titles)
actual = folds.drop_duplicates("snapshot_date")
fig.add_trace(
    line(actual["snapshot_date"], actual["observed_rate"], "observed", GREY, width=5), 1, 1
)
for name, part in folds.groupby("model", sort=False):
    predicted = line(
        part["snapshot_date"], part["predicted_rate"], f"{name} predicted", COLORS[name]
    )
    fig.add_trace(predicted, 1, 1)
    fig.add_trace(
        line(part["snapshot_date"], part["auc"], name, COLORS[name], showlegend=False), 1, 2
    )
fig.update_yaxes(tickformat=".0%", rangemode="tozero", row=1, col=1)
fig.update_xaxes(title_text="snapshot date")
show(fig, "Rolling-origin backtest", height=380)

errors = folds.assign(rate_error=folds["predicted_rate"] - folds["observed_rate"])
errors["abs_rate_error"] = errors["rate_error"].abs()
display(errors.groupby("model")[["rate_error", "abs_rate_error", "auc", "brier"]].mean(), folds)

,rate_error,abs_rate_error,auc,brier
model,,,,
aft,-0.0194,0.0194,0.8636,0.1105
cox,0.0089,0.0089,0.8633,0.1105


,snapshot_date,model,train_rows,players,predicted_rate,observed_rate,auc,brier
0,2025-07-01,cox,39048,3899,0.2771,0.2729,0.8483,0.1161
1,2025-07-01,aft,39048,3899,0.2486,0.2729,0.8491,0.1161
2,2025-08-01,cox,42947,3947,0.2847,0.2731,0.8586,0.1160
3,2025-08-01,aft,42947,3947,0.2567,0.2731,0.8585,0.1153
4,2025-09-01,cox,46894,3995,0.2794,0.2668,0.8551,0.1138
5,2025-09-01,aft,46894,3995,0.2505,0.2668,0.8536,0.1140
6,2025-10-01,cox,50889,4037,0.2757,0.2633,0.8820,0.1048
7,2025-10-01,aft,50889,4037,0.2490,0.2633,0.8841,0.1041
8,2025-11-01,cox,54926,4103,0.2698,0.2596,0.8701,0.1089
9,2025-11-01,aft,54926,4103,0.2410,0.2596,0.8695,0.1088


## 14. Production model: refit on train + validation

The best configuration is refitted on the training and validation snapshots together,
with labels as known on the test snapshot date. The hyperparameters, including the
number of boosting rounds, are the tuned ones.

The production model has seen the validation data, so it cannot get a calibrator of its
own from it; it reuses the calibrator of the evaluation model. Step 15 checks on test
whether that transfer holds.

In [24]:
production_dates = config.TRAIN_SNAPSHOTS.append(pd.DatetimeIndex([config.VAL_SNAPSHOT]))
production_train = features.add_labels(snapshots[date.isin(production_dates)], config.TEST_SNAPSHOT)
production = make_model(best, tuned[best], SEED).fit(*xy(production_train))

sizes = {
    "evaluation (train)": describe(train),
    "production (train + val)": describe(production_train),
}
display(pd.DataFrame.from_dict(sizes, orient="index"))

# The production model is explained on the players it will score.
explained["production"] = explain(production, "production", scoring[FEATURES], SEED)
tables["feature_importance"] = pd.concat(
    [table for table, _ in explained.values()], ignore_index=True
)
tables["shap_values"] = pd.concat([shap for _, shap in explained.values()], ignore_index=True)
compared_models = tables["feature_importance"]["model"].isin([best, "production"])
importance_chart(
    tables["feature_importance"][compared_models],
    f"Feature importance: evaluation {best} (test sample) vs production (scoring sample)",
)

,rows,players,snapshots,event_rate
evaluation (train),63229,13852,18,0.3481
production (train + val),67464,14548,19,0.3652


## 15. Evaluation vs production

Both models predict the same test snapshot. The production model has not seen it either
(its labels stop at the test snapshot date), so the comparison is fair: the only
difference is the three extra months of training data.

In [25]:
production_raw = production.predict_survival(X_test, TIMES)
compared = {
    ("evaluation", "raw"): test_surv[(best, "raw")],
    ("evaluation", "calibrated"): test_surv[(best, "calibrated")],
    ("production", "raw"): production_raw,
    ("production", "calibrated"): calibrators[best].transform(production_raw, TIMES),
}
tables["eval_vs_production"] = pd.DataFrame(
    [
        {
            "model": name,
            "variant": variant,
            **ev.summary_metrics(t_test, e_test, surv, TIMES, HORIZON),
            **ev.horizon_metrics(t_test, e_test, surv[:, K_HORIZON], HORIZON),
        }
        for (name, variant), surv in compared.items()
    ]
).drop(columns=["auc", "brier"])  # already reported as auc_horizon / brier_horizon
tables["eval_vs_production_curves"] = pd.concat(
    [
        ev.survival_curve_table(t_test, e_test, surv, TIMES).assign(model=name, variant=variant)
        for (name, variant), surv in compared.items()
    ],
    ignore_index=True,
)
tables["eval_vs_production_scores"] = pd.DataFrame(
    {
        "player_id": test["player_id"].to_numpy(),
        "churned": e_test & (t_test <= HORIZON),
        **{
            f"{name}_{variant}": 1 - surv[:, K_HORIZON]
            for (name, variant), surv in compared.items()
        },
    }
)
risk_correlation = spearmanr(
    models[best].predict_risk(X_test), production.predict_risk(X_test)
).statistic
churn_prob_gap = np.abs(
    compared[("evaluation", "calibrated")][:, K_HORIZON]
    - compared[("production", "calibrated")][:, K_HORIZON]
).mean()
agreement = {
    "spearman_risk_score": float(risk_correlation),
    "mean_abs_diff_churn_prob": float(churn_prob_gap),
}
display(tables["eval_vs_production"], pd.Series(agreement, name="agreement on test").to_frame())

,model,variant,c_index,ibs,brier_horizon,auc_horizon,predicted_rate,observed_rate
0,evaluation,raw,0.8199,0.0719,0.1093,0.8624,0.2741,0.258
1,evaluation,calibrated,0.8204,0.0714,0.1094,0.8617,0.2633,0.258
2,production,raw,0.8209,0.0720,0.1093,0.8628,0.2746,0.258
3,production,calibrated,0.8211,0.0715,0.1098,0.8626,0.2647,0.258


,agreement on test
spearman_risk_score,0.9936
mean_abs_diff_churn_prob,0.0121


In [26]:
titles = [
    "mean predicted survival vs Kaplan-Meier",
    f"calibrated P(churn within {HORIZON} days), one point per player",
]
fig = make_subplots(rows=1, cols=2, subplot_titles=titles)
curves = tables["eval_vs_production_curves"]
observed = curves.drop_duplicates("day")
fig.add_trace(
    line(observed["day"], observed["observed"], "observed (Kaplan-Meier)", GREY, width=5), 1, 1
)
role_colors = {"evaluation": COLORS[best], "production": COLORS["production"]}
for (name, variant), part in curves.groupby(["model", "variant"], sort=False):
    label = f"{name} {variant}"
    fig.add_trace(
        line(part["day"], part["predicted"], label, role_colors[name], DASHES[variant]), 1, 1
    )
probabilities = tables["eval_vs_production_scores"]
players_on_test = go.Scatter(
    x=probabilities["evaluation_calibrated"],
    y=probabilities["production_calibrated"],
    mode="markers",
    marker={"size": 4, "opacity": 0.4, "color": COLORS["production"]},
    showlegend=False,
    hoverinfo="skip",
)
fig.add_trace(players_on_test, 1, 2)
fig.add_trace(line([0, 1], [0, 1], "y = x", GREY, "dash", showlegend=False), 1, 2)
fig.update_xaxes(title_text="days since snapshot", row=1, col=1)
fig.update_yaxes(tickformat=".0%", row=1, col=1)
fig.update_xaxes(title_text="evaluation model", row=1, col=2)
fig.update_yaxes(title_text="production model", row=1, col=2)
show(fig, "Evaluation vs production model on the test snapshot")

## 16. Riskiest players

The production model scores every player at risk on the scoring snapshot (the day after
the data end). Players are ranked by the calibrated probability of churning within 60
days; calibration creates ties, which the raw risk score breaks. Risk segments are fixed
thresholds on that probability (`config.RISK_SEGMENTS`).

Expect the very top of the list to be players who are days away from the 30-day
inactivity threshold: their churn is close to a matter of definition. Step 17 therefore
also looks at the riskiest players among those who were active in the last week.

In [27]:
scores, score_curves = score_players(production, calibrators[best], scoring)
score_curves.insert(0, "player_id", scores["player_id"])
tables["scores"] = scores
tables["score_curves"] = score_curves
tables["top_risk_users"] = scores.head(config.TOP_N_RISKY)

segments = [name for name, _ in config.RISK_SEGMENTS]
segment_sizes = scores["risk_segment"].value_counts().reindex(segments, fill_value=0)
titles = ["players per risk segment", f"P(churn within {HORIZON} days)"]
fig = make_subplots(rows=1, cols=2, subplot_titles=titles)
segment_bars = go.Bar(
    x=segments,
    y=segment_sizes,
    text=segment_sizes,
    marker_color=[COLORS[name] for name in segments],
)
fig.add_trace(segment_bars, 1, 1)
histogram = go.Histogram(
    x=scores[f"churn_prob_{HORIZON}d"],
    xbins={"start": 0, "end": 1.0001, "size": 0.025},
    marker_color=COLORS["production"],
)
fig.add_trace(histogram, 1, 2)
fig.update_xaxes(tickformat=".0%", row=1, col=2)
title = f"Scoring snapshot {config.SCORING_SNAPSHOT.date()}: {len(scores):,} players at risk"
show(fig, title, height=340, showlegend=False)

print(f"Segment thresholds on P(churn within {HORIZON} days): {dict(config.RISK_SEGMENTS)}")
shown = ["risk_rank", "player_id", f"churn_prob_{HORIZON}d", "rul_days", "risk_segment"]
shown += ["recency_days", "active_days_30d", "tenure_days"]
display(scores[shown].head(20))

Segment thresholds on P(churn within 60 days): {'high': 0.7, 'medium': 0.4, 'low': 0.0}


,risk_rank,player_id,churn_prob_60d,rul_days,risk_segment,recency_days,active_days_30d,tenure_days
0,1,13237,1.0,1.0000,high,30,1,55
1,2,6034,1.0,1.0000,high,30,1,48
2,3,15585,1.0,1.0000,high,30,1,133
3,4,7305,1.0,1.0000,high,30,1,82
4,5,7764,1.0,1.0000,high,30,1,78
5,6,853,1.0,1.0000,high,30,1,34
6,7,5814,1.0,1.0000,high,30,1,30
7,8,13329,1.0,1.0000,high,30,1,60
8,9,438,1.0,1.0000,high,30,1,67
9,10,14594,1.0,1.0000,high,30,1,286


## 17. Scoring of the riskiest players

The scorecard of a player holds:

- the calibrated probability of churning within 7, 14, 30 and 60 days;
- the remaining useful life in days (capped at 85);
- the risk segment;
- the three features that push the player's risk up the most, with their SHAP values on
  the risk-score scale;
- the individual survival curve.

Read the first weeks of an individual curve with care. A player last active `r` days
before the snapshot cannot churn before day `31 - r`, but neither model can express a
player-specific dead time: Cox shares one baseline hazard across players, AFT one noise
distribution, and the calibrator works day by day. A curve can therefore start to fall
earlier than is possible for that player. By day 60 every scored player can have
churned, so the 60-day probability has no such constraint.

In [28]:
probability_columns = [f"churn_prob_{h}d" for h in config.SCORE_HORIZONS]
drivers = ["driver_1", "driver_2", "driver_3"]
scorecard = ["risk_rank", "player_id", "risk_segment", *probability_columns, "rul_days"]
scorecard += ["recency_days", *drivers]

print(f"Top {config.TOP_N_RISKY} players are saved as `top_risk_users`; the first 15:")
display(tables["top_risk_users"][scorecard].head(15))

recently_active = scores[scores["recency_days"] <= 7]
print("Riskiest players among those active in the last 7 days:")
display(recently_active[scorecard].head(15))

Top 100 players are saved as `top_risk_users`; the first 15:


,risk_rank,player_id,risk_segment,churn_prob_7d,churn_prob_14d,churn_prob_30d,churn_prob_60d,rul_days,recency_days,driver_1,driver_2,driver_3
0,1,13237,high,1.0,1.0,1.0,1.0,1.0,30,recency_days,active_days_30d,active_days_7d
1,2,6034,high,1.0,1.0,1.0,1.0,1.0,30,recency_days,active_days_30d,tenure_days
2,3,15585,high,1.0,1.0,1.0,1.0,1.0,30,recency_days,active_days_30d,active_days_7d
3,4,7305,high,1.0,1.0,1.0,1.0,1.0,30,recency_days,active_days_30d,active_days_7d
4,5,7764,high,1.0,1.0,1.0,1.0,1.0,30,recency_days,active_days_30d,active_days_7d
5,6,853,high,1.0,1.0,1.0,1.0,1.0,30,recency_days,active_days_30d,tenure_days
6,7,5814,high,1.0,1.0,1.0,1.0,1.0,30,recency_days,active_days_30d,tenure_days
7,8,13329,high,1.0,1.0,1.0,1.0,1.0,30,recency_days,active_days_30d,active_days_7d
8,9,438,high,1.0,1.0,1.0,1.0,1.0,30,recency_days,active_days_30d,active_days_7d
9,10,14594,high,1.0,1.0,1.0,1.0,1.0,30,recency_days,active_days_30d,active_days_7d


Riskiest players among those active in the last 7 days:


,risk_rank,player_id,risk_segment,churn_prob_7d,churn_prob_14d,churn_prob_30d,churn_prob_60d,rul_days,recency_days,driver_1,driver_2,driver_3
431,432,19264,high,0.0,0.0,0.7500,0.8485,32.6011,4,activity_ratio_7d_30d,active_days_7d,acquisition_channel
436,437,10736,high,0.0,0.0,0.7500,0.8485,32.6099,6,activity_ratio_7d_30d,active_days_7d,sessions_30d
438,439,1088,high,0.0,0.0,0.7500,0.8485,32.6099,5,active_days_30d,active_days_7d,acquisition_channel
442,443,1998,high,0.0,0.0,0.7500,0.8485,32.6099,5,active_days_30d,acquisition_channel,tenure_days
448,449,7022,high,0.0,0.0,0.7500,0.8485,32.6099,7,active_days_30d,tenure_days,active_days_7d
451,452,1250,high,0.0,0.0,0.7500,0.8485,32.6099,5,active_days_30d,tenure_days,active_days_7d
453,454,14102,high,0.0,0.0,0.7500,0.8485,32.6145,6,tenure_days,active_days_7d,acquisition_channel
459,460,16411,high,0.0,0.0,0.6667,0.8485,34.1768,6,active_days_30d,acquisition_channel,active_days_7d
465,466,17306,high,0.0,0.0,0.6667,0.8485,34.2794,6,tenure_days,active_days_7d,active_days_30d
472,473,19533,high,0.0,0.0,0.6667,0.8485,34.3701,7,active_days_7d,tenure_days,acquisition_channel


In [29]:
survival = score_curves.set_index("player_id")
days = [0, *TIMES]
average = line(days, [1.0, *survival.mean()], "average of all scored players", GREY, "dash", 4)
fig = go.Figure(average)
# Calibration gives neighbours in the ranking the same curve, so show one player per segment.
examples = recently_active.groupby("risk_segment", sort=False).head(1)
for player_id, segment in zip(examples["player_id"], examples["risk_segment"], strict=True):
    label = f"player {player_id} ({segment} risk)"
    fig.add_trace(line(days, [1.0, *survival.loc[player_id]], label, COLORS[segment]))
fig.add_vline(x=HORIZON, line_dash="dot", line_color=GREY)
show(
    fig,
    "Calibrated survival curves: the riskiest recently active player of each segment",
    "days since the scoring snapshot",
    "probability of not having churned",
    yaxis_tickformat=".0%",
    yaxis_range=[0, 1.02],
)

player = recently_active.iloc[0]
print(f"Risk drivers of player {player['player_id']} (rank {player['risk_rank']}):")
display(
    pd.DataFrame(
        {
            "feature": [player[name] for name in drivers],
            "value": [player[player[name]] for name in drivers],
            "shap": [player[f"{name}_shap"] for name in drivers],
        }
    )
)

Risk drivers of player 19264 (rank 432):


,feature,value,shap
0,activity_ratio_7d_30d,0.2143,0.5941
1,active_days_7d,1,0.5919
2,acquisition_channel,paid_ads,0.2866


## 18. Artifacts for the dashboard

Everything the dashboard reads is written to `artifacts/`:

- one parquet table per chart (the `tables` collected above);
- `metrics.json` with the configuration, dataset sizes, tuning results and headline
  numbers;
- the two evaluation models, the production model and the calibrators as JSON under
  `artifacts/models/`.

In [30]:
models_dir = ARTIFACTS_DIR / "models"
models_dir.mkdir(parents=True, exist_ok=True)
for name, table in tables.items():
    table.to_parquet(ARTIFACTS_DIR / f"{name}.parquet", index=False)
for name, model in (models | {"production": production}).items():
    model.save(models_dir / name)
for name, calibrator in calibrators.items():
    (models_dir / f"calibrator_{name}.json").write_text(json.dumps(calibrator.to_dict()))

train_span = (config.TRAIN_SNAPSHOTS[0], config.TRAIN_SNAPSHOTS[-1])
metadata = {
    "generated_at": datetime.now(UTC).isoformat(timespec="seconds"),
    "horizon_days": HORIZON,
    "inactivity_days": config.INACTIVITY_DAYS,
    "max_follow_up_days": config.MAX_FOLLOW_UP_DAYS,
    "time_grid": [int(TIMES[0]), int(TIMES[-1])],
    "features": FEATURES,
    "snapshots": {
        "train": [str(day.date()) for day in train_span],
        "val": str(config.VAL_SNAPSHOT.date()),
        "test": str(config.TEST_SNAPSHOT.date()),
        "scoring": str(config.SCORING_SNAPSHOT.date()),
    },
    "datasets": {
        "train": describe(train),
        "val": describe(val, full_follow_up=True),
        "test": describe(test, full_follow_up=True),
        "production_train": describe(production_train),
        "scoring": {"rows": len(scoring)},
    },
    "tuning": {"n_trials": N_TRIALS, "val_ibs": val_ibs, "best_params": tuned},
    "best_model": best,
    "n_bootstrap": N_BOOTSTRAP,
    "rul": rul_summary,
    "eval_vs_production_agreement": agreement,
    "risk_segments": dict(config.RISK_SEGMENTS),
    "seed": SEED,
}
(ARTIFACTS_DIR / "metrics.json").write_text(json.dumps(metadata, indent=2))

written = sorted(path for path in ARTIFACTS_DIR.rglob("*") if path.is_file())
listing = {
    "file": [str(path.relative_to(ARTIFACTS_DIR)) for path in written],
    "kB": [round(path.stat().st_size / 1024, 1) for path in written],
}
display(pd.DataFrame(listing))

,file,kB
0,backtest.parquet,5.9
1,bootstrap_differences.parquet,3.8
2,bootstrap_draws.parquet,33.8
3,bootstrap_summary.parquet,4.3
4,calibration.parquet,6.0
5,eval_vs_production.parquet,5.2
6,eval_vs_production_curves.parquet,8.0
7,eval_vs_production_scores.parquet,110.1
8,feature_importance.parquet,3.8
9,hazard.parquet,27.0


## 19. Dashboard

The Streamlit dashboard reads only `artifacts/` and shows every section of this notebook
interactively, in English (default) and Russian:

```bash
uv run streamlit run dashboard/app.py
```